In [ ]:
# Clone the repository if running in Google Colab or similar environment
!git clone https://github.com/yangzeha/LightGCL.git
%cd LightGCL

# LightGCL: Simple Yet Effective Graph Contrastive Learning for Recommendation

This notebook implements the LightGCL model for recommendation. It is based on the `main.py` script but adapted for interactive execution.

GitHub Repository: [https://github.com/yangzeha/LightGCL](https://github.com/yangzeha/LightGCL)

In [ ]:
import numpy as np
import torch
import pickle
from model import LightGCL
from utils import metrics, scipy_sparse_mat_to_torch_sparse_tensor, TrnData
import pandas as pd
from tqdm import tqdm
import time
import torch.utils.data as data
import os

# ========================== Configuration ==========================
class Args:
    def __init__(self):
        self.lr = 1e-3
        self.decay = 0.99
        self.batch = 256
        self.inter_batch = 4096
        self.note = None
        self.lambda1 = 0.2
        self.lambda2 = 1e-7
        self.epoch = 100
        self.d = 64
        self.q = 5
        self.gnn_layer = 2
        self.data = 'yelp'
        self.dropout = 0.0
        self.temp = 0.2
        self.cuda = '0'

args = Args()

# Check device
if torch.cuda.is_available():
    device = 'cuda:' + args.cuda
else:
    device = 'cpu'
print(f"Using device: {device}")

# hyperparameters
d = args.d
l = args.gnn_layer
temp = args.temp
batch_user = args.batch
epoch_no = args.epoch
max_samp = 40
lambda_1 = args.lambda1
lambda_2 = args.lambda2
dropout = args.dropout
lr = args.lr
decay = args.decay
svd_q = args.q

# ========================== Data Loading ==========================
path = 'data/' + args.data + '/'
print(f"Loading data from {path}...")
f = open(path+'trnMat.pkl','rb')
train = pickle.load(f)
train_csr = (train!=0).astype(np.float32)
f = open(path+'tstMat.pkl','rb')
test = pickle.load(f)
print('Data loaded.')
print('user_num:',train.shape[0],'item_num:',train.shape[1],'lambda_1:',lambda_1,'lambda_2:',lambda_2,'temp:',temp,'q:',svd_q)

epoch_user = min(train.shape[0], 30000)

# normalizing the adj matrix
rowD = np.array(train.sum(1)).squeeze()
colD = np.array(train.sum(0)).squeeze()
for i in range(len(train.data)):
    train.data[i] = train.data[i] / pow(rowD[train.row[i]]*colD[train.col[i]], 0.5)

# construct data loader
train = train.tocoo()
train_data = TrnData(train)
train_loader = data.DataLoader(train_data, batch_size=args.inter_batch, shuffle=True, num_workers=0)

adj_norm = scipy_sparse_mat_to_torch_sparse_tensor(train)
adj_norm = adj_norm.coalesce().to(torch.device(device))
print('Adj matrix normalized.')

# perform svd reconstruction
adj = scipy_sparse_mat_to_torch_sparse_tensor(train).coalesce().to(torch.device(device))
print('Performing SVD...')
svd_u,s,svd_v = torch.svd_lowrank(adj, q=svd_q)
u_mul_s = svd_u @ (torch.diag(s))
v_mul_s = svd_v @ (torch.diag(s))
del s
print('SVD done.')

# process test set
test_labels = [[] for i in range(test.shape[0])]
for i in range(len(test.data)):
    row = test.row[i]
    col = test.col[i]
    test_labels[row].append(col)
print('Test data processed.')

# ========================== Model & Training ==========================
loss_list = []
loss_r_list = []
loss_s_list = []
recall_20_x = []
recall_20_y = []
ndcg_20_y = []
recall_40_y = []
ndcg_40_y = []

model = LightGCL(adj_norm.shape[0], adj_norm.shape[1], d, u_mul_s, v_mul_s, svd_u.T, svd_v.T, train_csr, adj_norm, l, temp, lambda_1, lambda_2, dropout, batch_user, device)
#model.load_state_dict(torch.load('saved_model.pt'))
model.to(torch.device(device))
optimizer = torch.optim.Adam(model.parameters(),weight_decay=0,lr=lr)
#optimizer.load_state_dict(torch.load('saved_optim.pt'))

current_lr = lr

# Create directories if they don't exist
if not os.path.exists('log'):
    os.makedirs('log')
if not os.path.exists('saved_model'):
    os.makedirs('saved_model')

for epoch in range(epoch_no):
    if (epoch+1)%50 == 0:
        torch.save(model.state_dict(),'saved_model/saved_model_epoch_'+str(epoch)+'.pt')
        torch.save(optimizer.state_dict(),'saved_model/saved_optim_epoch_'+str(epoch)+'.pt')

    epoch_loss = 0
    epoch_loss_r = 0
    epoch_loss_s = 0
    train_loader.dataset.neg_sampling()
    
    # Wrap with tqdm for progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epoch_no}")
    for i, batch in enumerate(pbar):
        uids, pos, neg = batch
        uids = uids.long().to(torch.device(device))
        pos = pos.long().to(torch.device(device))
        neg = neg.long().to(torch.device(device))
        iids = torch.concat([pos, neg], dim=0)

        # feed
        optimizer.zero_grad()
        loss, loss_r, loss_s= model(uids, iids, pos, neg)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.cpu().item()
        epoch_loss_r += loss_r.cpu().item()
        epoch_loss_s += loss_s.cpu().item()

        if device.startswith('cuda'):
            torch.cuda.empty_cache()
        pbar.set_postfix({'Loss': loss.cpu().item()})

    batch_no = len(train_loader)
    epoch_loss = epoch_loss/batch_no
    epoch_loss_r = epoch_loss_r/batch_no
    epoch_loss_s = epoch_loss_s/batch_no
    loss_list.append(epoch_loss)
    loss_r_list.append(epoch_loss_r)
    loss_s_list.append(epoch_loss_s)
    print('Epoch:',epoch,'Loss:',epoch_loss,'Loss_r:',epoch_loss_r,'Loss_s:',epoch_loss_s)

    if epoch % 3 == 0:  # test every 3 epochs
        test_uids = np.array([i for i in range(adj_norm.shape[0])])
        batch_no_test = int(np.ceil(len(test_uids)/batch_user))

        all_recall_20 = 0
        all_ndcg_20 = 0
        all_recall_40 = 0
        all_ndcg_40 = 0
        for batch in tqdm(range(batch_no_test), desc="Testing"):
            start = batch*batch_user
            end = min((batch+1)*batch_user,len(test_uids))

            test_uids_input = torch.LongTensor(test_uids[start:end]).to(torch.device(device))
            predictions = model(test_uids_input,None,None,None,test=True)
            predictions = np.array(predictions.cpu())

            #top@20
            recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
            #top@40
            recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

            all_recall_20+=recall_20
            all_ndcg_20+=ndcg_20
            all_recall_40+=recall_40
            all_ndcg_40+=ndcg_40
            
        print('-------------------------------------------')
        print('Test of epoch',epoch,':','Recall@20:',all_recall_20/batch_no_test,'Ndcg@20:',all_ndcg_20/batch_no_test,'Recall@40:',all_recall_40/batch_no_test,'Ndcg@40:',all_ndcg_40/batch_no_test)
        recall_20_x.append(epoch)
        recall_20_y.append(all_recall_20/batch_no_test)
        ndcg_20_y.append(all_ndcg_20/batch_no_test)
        recall_40_y.append(all_recall_40/batch_no_test)
        ndcg_40_y.append(all_ndcg_40/batch_no_test)

# ========================== Final Evaluation ==========================
test_uids = np.array([i for i in range(adj_norm.shape[0])])
batch_no_test = int(np.ceil(len(test_uids)/batch_user))

all_recall_20 = 0
all_ndcg_20 = 0
all_recall_40 = 0
all_ndcg_40 = 0
for batch in range(batch_no_test):
    start = batch*batch_user
    end = min((batch+1)*batch_user,len(test_uids))

    test_uids_input = torch.LongTensor(test_uids[start:end]).to(torch.device(device))
    predictions = model(test_uids_input,None,None,None,test=True)
    predictions = np.array(predictions.cpu())

    #top@20
    recall_20, ndcg_20 = metrics(test_uids[start:end],predictions,20,test_labels)
    #top@40
    recall_40, ndcg_40 = metrics(test_uids[start:end],predictions,40,test_labels)

    all_recall_20+=recall_20
    all_ndcg_20+=ndcg_20
    all_recall_40+=recall_40
    all_ndcg_40+=ndcg_40

print('-------------------------------------------')
print('Final test:','Recall@20:',all_recall_20/batch_no_test,'Ndcg@20:',all_ndcg_20/batch_no_test,'Recall@40:',all_recall_40/batch_no_test,'Ndcg@40:',all_ndcg_40/batch_no_test)

recall_20_x.append('Final')
recall_20_y.append(all_recall_20/batch_no_test)
ndcg_20_y.append(all_ndcg_20/batch_no_test)
recall_40_y.append(all_recall_40/batch_no_test)
ndcg_40_y.append(all_ndcg_40/batch_no_test)

metric = pd.DataFrame({
    'epoch':recall_20_x,
    'recall@20':recall_20_y,
    'ndcg@20':ndcg_20_y,
    'recall@40':recall_40_y,
    'ndcg@40':ndcg_40_y
})
current_t = time.gmtime()
metric.to_csv('log/result_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.csv')

torch.save(model.state_dict(),'saved_model/saved_model_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.pt')
torch.save(optimizer.state_dict(),'saved_model/saved_optim_'+args.data+'_'+time.strftime('%Y-%m-%d-%H',current_t)+'.pt')

## Configuration

Set the hyperparameters here. You can change `data` to 'gowalla', 'yelp', etc.

## Data Loading

## Data Preprocessing & SVD

## Model Initialization

## Training Loop

## Final Evaluation & Saving Results